# Bagging - Random Forest Classifier

scaling technique is not required for tree based algortihm, 
only encodng can be used 

In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline



In [4]:
df = pd.read_csv("../datasets/loan_data.csv")
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [5]:
X = df.drop(columns = "loan_status")
y = df['loan_status']
X.shape

(45000, 13)

In [6]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
cat_cols = ["person_gender", "person_education", "person_home_ownership", "loan_intent", "previous_loan_defaults_on_file"]

for i in cat_cols:
    print(df[i].unique())

<StringArray>
['female', 'male']
Length: 2, dtype: str
<StringArray>
['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
Length: 5, dtype: str
<StringArray>
['RENT', 'OWN', 'MORTGAGE', 'OTHER']
Length: 4, dtype: str
<StringArray>
[         'PERSONAL',         'EDUCATION',           'MEDICAL',
           'VENTURE',   'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
Length: 6, dtype: str
<StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [ ]:
preprocessing = ColumnTransformer(
    transformers = [
        ('one_hot_encoding', OneHotEncoder(sparse_output = False, handle_unknown="ignore"), [1,2,5,7,12]),
        # ('ordinal_encode', OrdinalEncoder(categories=[
        #     ['female', 'male'], 
        #     ['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'], 
        #     ['RENT', 'OWN', 'MORTGAGE', 'OTHER'], 
        #     ['PERSONAL','EDUCATION' ,'MEDICAL','VENTURE','HOMEIMPROVEMENT','DEBTCONSOLIDATION'], 
        #     ['No', 'Yes']
        # ]), 
        # ["person_gender", "person_education", "person_home_ownership", "loan_intent", "previous_loan_defaults_on_file"])                                                                               # IT  ( CATEGORY )ACCEPTS 2D
    ],
    remainder="passthrough"
)

# i commented the ordinal encoder , because column transformer executes parallely and it gives two diffrent and separet encoded values for the xtrain, so we get duplicate values when we use two diffrent encoders for the same column
# if categorical_column has order use OrdinalEncoder, else use OneHotEncoder that is best for all

# so we have to use any one of the encoding if we apply for same columnsgit add . 


pipe = Pipeline([
    ('preprocess', preprocessing),
    ('model', RandomForestClassifier())
])

pipe.fit(Xtrain, ytrain)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('one_hot_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

### Why we dont have to pass updated xtrain while training when we use pipeline? 

```Xtrain and Xtest do not need to be manually updated when you use a Pipeline.```

**When you do:**
> pipe.fit(Xtrain, ytrain)

_this happens internally:_
- preprocess.fit_transform(Xtrain) runs first
- The transformed training data is sent to RandomForestClassifier.fit(...)

**Then when you do:**
> y_pred_train = pipe.predict(Xtrain)
> y_pred_test = pipe.predict(Xtest)

_the pipeline again does this internally:_
- preprocess.transform(Xtrain) or preprocess.transform(Xtest)
- Sends the transformed data to model.predict(...)

In [13]:
# if we want to see the internally transformed xtrain and xtest data
Xtrain_transformed = pipe.named_steps['preprocess'].transform(Xtrain)
Xtest_transformed = pipe.named_steps['preprocess'].transform(Xtest)

# print(Xtrain_transformed)
# print(Xtest_transformed)

assignment:  use callsification report evaluation for bot trian and
conslude the model is goodfit to overfit or underfit 

## Model evaluation steps

In [24]:
y.value_counts()

loan_status
0    35000
1    10000
Name: count, dtype: int64

In [16]:
y_pred_train = pipe.predict(Xtrain)
y_pred_test = pipe.predict(Xtest)

print(y_pred_test)
print(y_pred_train)

[0 0 1 ... 0 0 0]
[0 0 0 ... 0 1 0]


In [20]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

train_acc = accuracy_score(ytrain, y_pred_train)
test_acc = accuracy_score(ytest, y_pred_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 1.0
Test Accuracy: 0.9286666666666666


In [21]:
print("Confusion Matrix:")
print(confusion_matrix(ytest, y_pred_test))

print("\nClassification Report:")
print(classification_report(ytest, y_pred_test))

Confusion Matrix:
[[6806  184]
 [ 458 1552]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95      6990
           1       0.89      0.77      0.83      2010

    accuracy                           0.93      9000
   macro avg       0.92      0.87      0.89      9000
weighted avg       0.93      0.93      0.93      9000



**Conclusion (confusion matrix):**
- 6806 class 0 cases were predicted correctly
- 1552 class 1 cases were predicted correctly
- 458 actual 1s were missed
- 184 actual 0s were wrongly predicted as 1

> So the model is good, but it is slightly weaker on the minority class than on the majority class, which is common in imbalanced data.



**Conclusion (classification_report):**
- The RandomForestClassifier is performing well overall on the test data. 
- It achieved about **93% accuracy**, with strong results for class 0 and reasonably good results for class 1. 
- Since this is an imbalanced dataset, the more important part is that the model still predicts the minority class fairly well, with **precision = 0.89, recall = 0.77, and f1-score = 0.83** for class 1. That means the model is able to identify many positive cases, but it is still missing some of them.

In [23]:
y_prob_test = pipe.predict_proba(Xtest)
print(y_prob_test)


# Meaning:
# first number = probability of class 0
# second number = probability of class 1
# So:
# predict() gives final class labels
# predict_proba() gives probabilities

[[0.95 0.05]
 [1.   0.  ]
 [0.18 0.82]
 ...
 [0.88 0.12]
 [0.64 0.36]
 [0.52 0.48]]


**Final Conclusion  (About good fit, underfit, or overfit):**

- The Random Forest model is not underfitting because it performs very well on both training and test data.
- However, since the training accuracy is 100% and the test accuracy is 92.87%, the model shows slight to moderate overfitting.
- So, the model is good in performance, but it is not a perfect good fit.

## if we want to reduce overfitting we can try 
RandomForestClassifier(max_depth=10, min_samples_split=10, min_samples_leaf=5, random_state=42)

**this is only hyper parameter tuning**